# HW — Topic 5 · 엔트로피로 Wordle 첫 단어 고르기

**확률통계 · Topic 5** | 선택 과제 · **가산점 5점** | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T05_bonus.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T05_bonus.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

---

이번 주는 **정식 과제가 없다.** 아래는 관심 있는 학생을 위한 **가산점 과제**다.
랩 Part 1 의 `entropy()` 를 그대로 쓴다.

> **Wordle 이란?** 숨겨진 **5글자 영어 단어**를 6번 안에 맞히는 온라인 단어 퍼즐이다.
> 매번 5글자 단어를 하나 추측해 넣으면 글자마다 힌트가 나온다 —
> 🟩 자리·글자 모두 맞음 / 🟨 그 글자는 있지만 자리가 틀림 / ⬜ 그 글자는 없음.
> 이 힌트로 후보를 좁혀 간다. (2021년부터 뉴욕타임스가 운영 · 하루 한 문제)
> 이 과제는 **"첫 추측을 어떤 단어로 하면 후보가 가장 많이 줄어드는가"** 를 엔트로피로 따진다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| (b) | 추측 단어 하나의 **정보량(엔트로피)** 을 재는 함수 완성 | 1 |
| (c) | 후보 단어들의 엔트로피 **순위** 출력 | 1 |
| (d) | 상·하위 단어의 **공통점** + **게임 전략** 서술 | 3 |

> (a) 패턴 함수와 (b)~(c) 를 잇는 히스토그램 그림은 아래 Part 0 · 「패턴 히스토그램」에
> 이미 주어져 있다. 손으로 채우는 칸은 **TODO 1~2 두 곳뿐**이고, 점수의 대부분은 (d) 해석이다.

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.

## Part 0. 준비 — 그대로 실행

- 후보 단어 목록 `WORDS` (21개) 와 **패턴 함수 `pattern()`** 이 주어진다.
- 한 번의 추측 결과는 각 글자마다 🟩(2, 정확한 위치) / 🟨(1, 다른 위치) / ⬜(0, 없음).
  → 결과는 $3^5 = 243$ 가지 패턴 중 하나다.
- 중복 글자 처리는 신경 쓰지 않는다 (간이 버전).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# ⚠️ 이 목록·시드는 바꾸지 마세요
WORDS = ["crane", "slate", "trace", "adieu", "audio", "raise", "roast",
         "stare", "least", "point", "movie", "beach", "chair", "dance",
         "fruit", "ghost", "house", "juice", "knife", "lemon", "mouse"]


def pattern(guess, answer):
    """guess 를 answer 에 맞춰본 결과. 🟩=2, 🟨=1, ⬜=0 을 5자리 튜플로 반환."""
    out = []
    for i, ch in enumerate(guess):
        if ch == answer[i]:
            out.append(2)
        elif ch in answer:
            out.append(1)
        else:
            out.append(0)
    return tuple(out)


def entropy(probs):
    """랩 Part 1 과 같은 엔트로피 (bits)."""
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


print("후보 단어", len(WORDS), "개")
print("예) pattern('crane', 'trace') =", pattern("crane", "trace"))

## `Counter` 로 패턴을 센다 — 이 문제의 핵심 도구

(b) 에서 할 일은 **"어떤 추측이 정답 후보들을 몇 개의 패턴으로 어떻게 쪼개는가"** 를 세는 것이다.
이때 `collections.Counter` 를 쓴다 (Part 0 에서 이미 import 했다).

**`Counter` 란?** 넣어 준 값들을 훑어 `{값: 등장 횟수}` 로 만들어 주는 특별한 `dict` 다.

```python
Counter(['a', 'b', 'a', 'c', 'a'])      # -> Counter({'a': 3, 'b': 1, 'c': 1})
```

**여기서는 "피드백 패턴"을 센다.** `pattern(guess, answer)` 는 5글자 힌트를 길이 5짜리
튜플로 돌려준다 (🟩=2 · 🟨=1 · ⬜=0). 예: `pattern('crane', 'trace')` → `(1, 2, 2, 0, 2)`.

```python
Counter(pattern('crane', a) for a in WORDS)
# -> Counter({(2,2,2,2,2): 1, (0,0,2,0,2): 1, (1,2,2,0,2): 1, ...})
#    "후보 중 이 패턴을 주는 단어가 몇 개인가" 를 패턴마다 센 것
```

- 튜플은 **불변**이라 `Counter`/`dict` 의 key 로 쓸 수 있다 (`pattern()` 이 리스트가 아니라
  튜플을 돌려주는 이유다).
- `(pattern(...) for a in WORDS)` 는 **제너레이터 식** — 리스트를 먼저 만들지 않고
  하나씩 흘려보내며 센다.
- `counts.values()` 각 패턴의 개수 · `sum(counts.values())` 전체 후보 수(= `len(WORDS)`) ·
  `counts.keys()` 관측된 서로 다른 패턴들.
- `Counter` 없이 쓰면 이 세 줄과 같다:

```python
counts = {}
for a in WORDS:
    pat = pattern('crane', a)
    counts[pat] = counts.get(pat, 0) + 1     # 없던 key 는 0 에서 시작
```

각 개수를 전체 후보 수로 나누면 **패턴에 대한 확률분포** `p(pattern)` 가 되고,
그 분포의 엔트로피가 이 추측의 정보량이다. **서로 다른 패턴으로 고르게 흩어질수록** 엔트로피가 크다.

## 문제 (b) — 추측 단어의 정보량 (1점)

추측 단어 `guess` 를 **모든 정답 후보** `answers` 에 대해 맞춰본다.
같은 패턴이 나온 후보끼리 묶으면, 패턴별 후보 수 → **패턴의 확률분포**가 된다.
그 분포의 엔트로피가 이 추측의 정보량이다.

$$H(\text{guess}) = -\sum_{\text{pattern}} p(\text{pattern}) \log_2 p(\text{pattern})$$

> 💡 좋은 추측 = 후보를 **가장 고르게 쪼개는** 추측. 고르게 쪼갤수록 $H$ 가 크고,
> 남는 후보가 가장 많이 줄어든다. 슬라이드의 "스무고개"가 이 이야기였다.

### ✏️ TODO 1 — `guess_entropy(guess, answers)` 를 완성한다

In [ ]:
def guess_entropy(guess, answers):
    # 각 정답 후보에 대해 pattern(guess, answer) 을 구해 패턴별 개수를 센다
    counts = Counter(pattern(guess, a) for a in answers)
    total = sum(counts.values())

    # TODO 1: 패턴별 확률 리스트를 만들고 entropy(...) 로 정보량을 돌려주세요
    #         힌트 - probs = [n / total for n in counts.values()] ;  return entropy(probs)
    return 0.0


print(f"H(crane) = {guess_entropy('crane', WORDS):.3f} bits")
print(f"H(lemon) = {guess_entropy('lemon', WORDS):.3f} bits")

## 패턴 히스토그램 — 좋은 추측 vs 나쁜 추측  (그대로 실행)

`Counter` 가 만든 **패턴별 후보 수**를 막대그래프로 그린다. 엔트로피가 높은 `crane` 과
낮은 `lemon` 을 나란히 놓고, "고르게 쪼갠다"가 그림에서 어떻게 보이는지 확인한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)

for ax, g in zip(axes, ["crane", "lemon"]):
    counts = Counter(pattern(g, a) for a in WORDS)
    sizes = sorted(counts.values(), reverse=True)      # 패턴별 후보 수, 큰 것부터
    H = entropy([s / len(WORDS) for s in sizes])
    ax.bar(range(len(sizes)), sizes, color="#3b4fd8", edgecolor="white")
    ax.set_title(f"guess = '{g}'   ({len(sizes)} patterns,  H = {H:.2f} bits)")
    ax.set_xlabel("feedback pattern (sorted by size)")

axes[0].set_ylabel("number of candidate words")
plt.tight_layout()
plt.show()

- **`crane`** — 막대가 **많고 대부분 높이 1**이다. 후보 21개가 거의 서로 다른 패턴으로
  흩어진다 → 한 번 추측하면 대부분 구별된다. (분포가 균등에 가깝다 → $H$ 큼)
- **`lemon`** — 막대가 **적고 몇 개가 우뚝하다**. 여러 후보가 같은 패턴에 몰린다 →
  추측해도 후보가 덜 줄어든다. (분포가 한쪽으로 쏠린다 → $H$ 작음)

**고르게 퍼진 히스토그램일수록 엔트로피가 크다** — 이것이 (b) 에서 계산한 값의 정체다.

## 문제 (c) — 순위 출력 (1점)

`WORDS` 의 모든 단어에 대해 `guess_entropy` 를 구하고, **큰 순서로 정렬**해
상위 5개와 하위 3개를 출력한다.

### ✏️ TODO 2 — 엔트로피로 정렬해 상·하위를 출력한다

In [ ]:
# TODO 2: (엔트로피, 단어) 쌍을 만들어 큰 순서로 정렬하세요
#         힌트 - sorted(((guess_entropy(w, WORDS), w) for w in WORDS), reverse=True)
scores = []

print("=== 상위 5개 (정보량이 큰 첫 단어) ===")
for h, w in scores[:5]:
    print(f"  {w:8s} {h:.3f} bits")
print("=== 하위 3개 ===")
for h, w in scores[-3:]:
    print(f"  {w:8s} {h:.3f} bits")

## 문제 (d) — 해석 (3점)

**이 과제에서 가장 배점이 큰 부분이다.** (b)(c) 의 결과와 위 히스토그램을 근거로,
아래 세 가지를 각각 **2~3문장**으로 쓴다. 이 셀을 더블클릭해 작성한다.

1. **엔트로피가 높은 단어의 공통점** — `crane` · `raise` · `stare` 등.
   어떤 글자 구성(모음 수, 흔한 자음, 위치)이 후보를 잘 쪼개는가?
2. **엔트로피가 낮은 단어의 공통점** — `lemon` · `knife` · `movie` 등.
   왜 이런 단어는 히스토그램의 막대가 몇 개에 몰리는가?
3. **게임 전략** — 위 관찰을 Wordle 을 실제로 푸는 순서로 옮긴다.
   - 첫 추측은 어떤 기준으로 고르는가?
   - 두 번째 추측부터는 무엇이 달라지는가? (후보 목록이 줄어든 뒤에도 같은 기준을 쓰는가)
   - "글자를 맞히는 것"과 "후보를 줄이는 것" 중 무엇을 노려야 하는가?

> 💡 (b) 의 $H$ 는 "이 추측을 하면 **평균적으로 몇 bit 만큼 후보의 불확실성이 줄어드는가**" 다.
> 슬라이드 "스무고개" — 매번 후보를 반으로 자르는 질문이 최선 — 과 이어서 생각해 볼 것.

> **1. 높은 단어의 공통점:** (여기에 작성)
>
> **2. 낮은 단어의 공통점:** (여기에 작성)
>
> **3. 게임 전략:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `TODO 1~2` 를 모두 채웠다
- [ ] 문제 (d) 의 **세 가지(높은 단어 공통점 · 낮은 단어 공통점 · 게임 전략)를 작성했다**
- [ ] 패턴 히스토그램 그림이 출력에 남아 있다
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 파일명을 **`HW_학번_T05_bonus.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**

### 참고

- 강의 슬라이드 Topic 5 — "스무고개로 읽으면 더 쉽다"
- 3Blue1Brown, *Solving Wordle using information theory* (영상)
- 랩 노트북 `T05_lab.ipynb` Part 1 의 `entropy()`